# Smart Lender - Loan Approval Prediction System
### End-to-End Machine Learning Notebook

This notebook contains standard data science workflow steps:
1. **Data Understanding**: Loading the Analytics Vidhya Loan Prediction Dataset and checking statistics.
2. **Exploratory Data Analysis (EDA)**: Visualizing features and relationships.
3. **Data Preprocessing**: Handling missing values, categorical encoding, and feature scaling.
4. **Class Imbalance Correction**: Oversampling the minority class using SMOTE.
5. **Model Building & Evaluation**: Training and comparing Decision Tree, Random Forest, KNN, and XGBoost Classifiers.

## Step 1: Import Dependencies and Load Dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plots style
sns.set_theme(style="darkgrid")
%matplotlib inline

# Dataset remote source
DATASET_URL = "https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv"
df = pd.read_csv(DATASET_URL)
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

## Step 2: Data Understanding

In [ ]:
print("--- Dataset Information ---")
df.info()

print("\n--- Missing Values Count ---")
print(df.isnull().sum())

print("\n--- Descriptive Statistics for Numerical Features ---")
df.describe()

## Step 3: Exploratory Data Analysis

In [ ]:
# Visualizing the Target Class: Loan Status
plt.figure(figsize=(6, 4))
sns.countplot(x='Loan_Status', data=df, palette=['#3a86ff', '#ff006e'])
plt.title("Distribution of Loan Status (Target)", fontsize=14, fontweight='bold')
plt.show()

# Credit History vs Loan Status
plt.figure(figsize=(6, 4))
sns.countplot(x='Credit_History', hue='Loan_Status', data=df, palette=['#ff006e', '#3a86ff'])
plt.title("Loan Approval by Credit History", fontsize=14, fontweight='bold')
plt.show()

# Distribution of Income by Education
plt.figure(figsize=(8, 4))
sns.boxplot(x='Education', y='ApplicantIncome', data=df, palette='Set2')
plt.title("Applicant Income by Education Level", fontsize=14, fontweight='bold')
plt.yscale('log')
plt.show()

## Step 4: Data Preprocessing & Feature Engineering

In [ ]:
df_clean = df.drop(columns=['Loan_ID'], errors='ignore').copy()

# Define columns
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Credit_History', 'Property_Area']
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

# Impute missing values with Mode (categorical) and Median/Mode (numerical)
for col in cat_cols:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)

df_clean['LoanAmount'] = df_clean['LoanAmount'].fillna(df_clean['LoanAmount'].median())
df_clean['Loan_Amount_Term'] = df_clean['Loan_Amount_Term'].fillna(df_clean['Loan_Amount_Term'].mode()[0])

# Standardize values in categorical columns
df_clean['Dependents'] = df_clean['Dependents'].astype(str).str.replace('+', '', regex=False)

# Custom categorical mapping dictionary
encoding_mappings = {
    'Gender': {'Male': 1, 'Female': 0},
    'Married': {'Yes': 1, 'No': 0},
    'Dependents': {'0': 0, '1': 1, '2': 2, '3': 3},
    'Education': {'Graduate': 1, 'Not Graduate': 0},
    'Self_Employed': {'Yes': 1, 'No': 0},
    'Property_Area': {'Rural': 0, 'Semiurban': 1, 'Urban': 2},
    'Loan_Status': {'Y': 1, 'N': 0}
}

for col, mapping in encoding_mappings.items():
    df_clean[col] = df_clean[col].map(mapping).astype(int)

print("Check preprocessed dataset preview:")
df_clean.head()

## Step 5: Data Scaling and Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_clean.drop(columns=['Loan_Status'])
y = df_clean['Loan_Status']

# Fit scaler on numerical values
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# Train-test split (80-20 Stratified)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}, Validation set shape: {X_val.shape}")

## Step 6: Address Class Imbalance using SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

print(f"Target class count before SMOTE: {y_train.value_counts().to_dict()}")
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"Target class count after SMOTE: {y_train_res.value_counts().to_dict()}")

## Step 7: Train & Evaluate Classifiers

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'XGBoost': XGBClassifier(n_estimators=50, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    preds = model.predict(X_val)
    
    results[name] = {
        'Accuracy': accuracy_score(y_val, preds),
        'Precision': precision_score(y_val, preds),
        'Recall': recall_score(y_val, preds),
        'F1-Score': f1_score(y_val, preds)
    }

# Display Metrics in structured DataFrame
metrics_df = pd.DataFrame(results).T
metrics_df

## Step 8: Visualize Model Comparison

In [ ]:
metrics_df.plot(kind='bar', figsize=(10, 6))
plt.title("Algorithm Performance Comparison")
plt.ylabel("Score")
plt.xlabel("Model")
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()